# CLIP + AST + ontology grid search — development 6000

This notebook keeps the **CLIP + AST + ontology decision structure frozen** and uses the same external `Acoustic_ontology.ipynb`.

The ontology rules and inference stages are not changed. The grid varies only three numerical parameters that already exist in the frozen ontology method:

- AST event-confidence threshold: `0.05, 0.08, 0.1, 0.12`
- ontology fusion weight λ: `0.15, 0.2, 0.25, 0.3, 0.35, 0.4`
- CLIP protection margin: `0.1, 0.15, 0.2, 0.25, 0.3`

Total: **120 configurations**.

The following remain fixed:

- minimum ontology evidence = `0.15`
- minimum ontology advantage = `0.05`
- strong ontology evidence = `0.60`
- CLIP Top-3 candidate restriction
- evidence = confidence × persistence × semantic strength
- SUM ontology normalization
- candidate-specific CLIP protection

Ground truth is used only to rank configurations on the **6000 development set**. Do not run this grid search on the independent 3500 evaluation set.


## 1. Load the shared ontology

In [9]:
%run ../Acoustic_ontology.ipynb


Ontology rules: 22
Ontology event mappings: 22


## 2. Setup and search space

In [10]:
from pathlib import Path
from itertools import product
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# Project paths
PROJECT_ROOT = Path('C:\\Users\\22313906t\\Desktop\\TAU_project')
CLIP_PATH = PROJECT_ROOT / 'results_CLIP' / 'tau_6000_clip_predictions.csv'
AST_PATH = PROJECT_ROOT / 'results_AST_6000' / 'tau_all_ast_events.csv'
OUTPUT_DIR = PROJECT_ROOT / 'results_ONTOLOGY_GRID_SEARCH_6000'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LEADERBOARD_PATH = OUTPUT_DIR / 'ontology_grid_search_18_settings.csv'
BEST_PREDICTIONS_PATH = OUTPUT_DIR / 'ontology_grid_search_best_predictions.csv'
BEST_PER_CLASS_PATH = OUTPUT_DIR / 'ontology_grid_search_best_per_class.csv'
BEST_CONFIG_PATH = OUTPUT_DIR / 'ontology_grid_search_best_config.json'

# Dataset settings
TAU_CLASSES = [
    'airport',
    'bus',
    'metro',
    'metro station',
    'park',
    'public square',
    'shopping mall',
    'pedestrian street',
    'street traffic',
    'tram',
]
EXPECTED_TOTAL = 6000
N_PER_CLASS = 600
CLIP_SCORE_COLUMNS = {scene: scene.replace(' ', '_') + '_score' for scene in TAU_CLASSES}

# Grid search
# Only these three existing parameters are tuned.
# The decision rules and semantic gates remain unchanged.
MIN_EVENT_CONFIDENCE_VALUES = [0.05, 0.08, 0.1, 0.12]
ONTOLOGY_LAMBDA_VALUES = [0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
CLIP_PROTECTION_MARGIN_VALUES = [0.1, 0.15, 0.2, 0.25, 0.3]

# Fixed parameters
MIN_ONTOLOGY_EVIDENCE = 0.15
MIN_ONTOLOGY_ADVANTAGE = 0.05
STRONG_ONTOLOGY_EVIDENCE = 0.6

# Modes
NORMALIZATION_MODE = 'sum'
CLIP_PROTECTION_MODE = 'candidate'
GRID_SETTINGS = list(product(MIN_EVENT_CONFIDENCE_VALUES, ONTOLOGY_LAMBDA_VALUES, CLIP_PROTECTION_MARGIN_VALUES))
print('=' * 80)
print('TAU 6000 DEVELOPMENT — FROZEN ONTOLOGY GRID SEARCH')
print('=' * 80)
print('\nCLIP:')
print(CLIP_PATH)
print('\nAST:')
print(AST_PATH)
print('\nShared ontology:')
print('../Acoustic_ontology.ipynb')
print('\nOutput:')
print(OUTPUT_DIR)
print('\nGrid values:')
print('MIN_EVENT_CONFIDENCE:', MIN_EVENT_CONFIDENCE_VALUES)
print('ONTOLOGY_LAMBDA:', ONTOLOGY_LAMBDA_VALUES)
print('CLIP_PROTECTION_MARGIN:', CLIP_PROTECTION_MARGIN_VALUES)
print('\nGrid size:', len(GRID_SETTINGS))
print('Ontology normalization:', NORMALIZATION_MODE.upper())
print('CLIP protection mode:', CLIP_PROTECTION_MODE)
print('\nFixed thresholds:')
print('MIN_ONTOLOGY_EVIDENCE:', MIN_ONTOLOGY_EVIDENCE)
print('MIN_ONTOLOGY_ADVANTAGE:', MIN_ONTOLOGY_ADVANTAGE)
print('STRONG_ONTOLOGY_EVIDENCE:', STRONG_ONTOLOGY_EVIDENCE)


TAU 6000 DEVELOPMENT — FROZEN ONTOLOGY GRID SEARCH

CLIP:
C:\Users\22313906t\Desktop\TAU_project\results_CLIP\tau_6000_clip_predictions.csv

AST:
C:\Users\22313906t\Desktop\TAU_project\results_AST_6000\tau_all_ast_events.csv

Shared ontology:
../Acoustic_ontology.ipynb

Output:
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_GRID_SEARCH_6000

Grid values:
MIN_EVENT_CONFIDENCE: [0.05, 0.08, 0.1, 0.12]
ONTOLOGY_LAMBDA: [0.15, 0.2, 0.25, 0.3, 0.35, 0.4]
CLIP_PROTECTION_MARGIN: [0.1, 0.15, 0.2, 0.25, 0.3]

Grid size: 120
Ontology normalization: SUM
CLIP protection mode: candidate

Fixed thresholds:
MIN_ONTOLOGY_EVIDENCE: 0.15
MIN_ONTOLOGY_ADVANTAGE: 0.05
STRONG_ONTOLOGY_EVIDENCE: 0.6


In [11]:
# Shared ontology check
# The grid notebook uses only fields already present in the ontology
# used by the basic CLIP+AST ontology notebook:
# rule["name"]
# rule["relation"]
if 'RULE_LOOKUP' not in globals():
    raise RuntimeError('RULE_LOOKUP was not loaded. Run the %run ../Acoustic_ontology.ipynb cell first.')
if 'normalize_event' not in globals():
    raise RuntimeError('normalize_event was not loaded. Run the shared ontology notebook first.')
if not isinstance(RULE_LOOKUP, dict):
    raise TypeError('RULE_LOOKUP must be a dictionary.')
required_rule_fields = {'name', 'relation', 'targets'}
bad_rules = []
for event_name, rule in RULE_LOOKUP.items():
    if not isinstance(rule, dict):
        bad_rules.append((event_name, 'rule is not a dict'))
        continue
    missing = required_rule_fields - set(rule.keys())
    if missing:
        bad_rules.append((event_name, f'missing {sorted(missing)}'))
if bad_rules:
    print('First invalid ontology mappings:')
    for item in bad_rules[:20]:
        print(item)
    raise RuntimeError('Shared ontology is incompatible with the basic ontology notebook.')
relations = sorted({str(rule['relation']).upper() for rule in RULE_LOOKUP.values()})
print('Ontology mappings:', len(RULE_LOOKUP))
print('Relations:', relations)
print('\nGrid search does NOT require cue_group or counts_as_strong_support.')


Ontology mappings: 22
Relations: ['REFINE', 'SUPPORT']

Grid search does NOT require cue_group or counts_as_strong_support.


## 4. Load and prepare CLIP / AST results

In [12]:
# Load CLIP results
if not CLIP_PATH.exists():
    raise FileNotFoundError(f'Cannot find CLIP predictions:\n{CLIP_PATH}')
if not AST_PATH.exists():
    raise FileNotFoundError(f'Cannot find AST events:\n{AST_PATH}')
clip_df = pd.read_csv(CLIP_PATH)
required_clip_columns = {'clip_id', 'ground_truth', 'clip_top1', 'clip_top2', 'clip_top3'}
missing_clip_columns = required_clip_columns - set(clip_df.columns)
if missing_clip_columns:
    raise ValueError(f'Missing CLIP columns: {sorted(missing_clip_columns)}')
clip_df['clip_id'] = clip_df['clip_id'].astype(str)
clip_df['ground_truth'] = clip_df['ground_truth'].astype(str)
clip_df = clip_df.drop_duplicates(subset=['clip_id'], keep='last').reset_index(drop=True)
for scene, column in CLIP_SCORE_COLUMNS.items():
    if column not in clip_df.columns:
        raise ValueError(f'Missing CLIP score column: {column}')

# Verify development set
print('\n' + '=' * 80)
print('CLIP DEVELOPMENT-SET CHECK')
print('=' * 80)
print('Rows:', len(clip_df))
print('Unique clips:', clip_df['clip_id'].nunique())
if len(clip_df) != EXPECTED_TOTAL or clip_df['clip_id'].nunique() != EXPECTED_TOTAL:
    raise RuntimeError(f'Expected exactly {EXPECTED_TOTAL} unique CLIP rows.')
class_counts = clip_df['ground_truth'].value_counts().reindex(TAU_CLASSES, fill_value=0)
print('\nClass distribution:')
print(class_counts)
if not (class_counts == N_PER_CLASS).all():
    raise RuntimeError(f'Expected {N_PER_CLASS} clips per class.')

# Load AST results
ast_df = pd.read_csv(AST_PATH)
if 'clip_id' not in ast_df.columns:
    raise ValueError('AST file has no clip_id column.')
ast_df['clip_id'] = ast_df['clip_id'].astype(str)

# Detect event and score columns
EVENT_COL = None
for candidate in ['label', 'event', 'event_label', 'audio_event', 'class_name']:
    if candidate in ast_df.columns:
        EVENT_COL = candidate
        break
if EVENT_COL is None:
    raise ValueError(f'Cannot find AST event column. Available: {list(ast_df.columns)}')
SCORE_COL = None
for candidate in ['max_score', 'score', 'mean_score']:
    if candidate in ast_df.columns:
        SCORE_COL = candidate
        break
if SCORE_COL is None:
    raise ValueError(f'Cannot find AST score column. Available: {list(ast_df.columns)}')
print('\nAST event column:', EVENT_COL)
print('AST score column:', SCORE_COL)

# Same persistence handling as basic ontology
if 'persistence' not in ast_df.columns:
    if 'count' in ast_df.columns and 'num_chunks' in ast_df.columns:
        ast_df[
            'persistence'] = pd.to_numeric(ast_df['count'],
            errors='coerce') / pd.to_numeric(ast_df['num_chunks'],
            errors='coerce').replace(0,
            np.nan,
        )
        ast_df['persistence'] = ast_df['persistence'].fillna(0.0)
    else:
        raise ValueError('AST file needs either persistence or count + num_chunks.')

# Filter AST to development clips
clip_ids = set(clip_df['clip_id'])
ast_df = ast_df[ast_df['clip_id'].isin(clip_ids)].copy()
ast_matched_ids = set(ast_df['clip_id'])
missing_ast_ids = clip_ids - ast_matched_ids
print('\n' + '=' * 80)
print('AST / CLIP MATCH CHECK')
print('=' * 80)
print('CLIP clips:', len(clip_ids))
print('AST matched clips:', len(ast_matched_ids))
print('Missing AST clips:', len(missing_ast_ids))
if missing_ast_ids:
    print('\nFirst missing AST IDs:')
    for clip_id in sorted(missing_ast_ids)[:20]:
        print(clip_id)
    raise RuntimeError('AST results do not cover all 6000 CLIP clips.')

# Normalize / convert AST fields
ast_df['event_norm'] = ast_df[EVENT_COL].apply(normalize_event)
ast_df['event_score'] = pd.to_numeric(ast_df[SCORE_COL], errors='coerce').fillna(0.0)
ast_df[
    'persistence'] = pd.to_numeric(ast_df['persistence'],
    errors='coerce').fillna(0.0).clip(lower=0.0,
    upper=1.0,
)

# Index AST by CLIP
ast_groups = {clip_id: group.copy() for clip_id, group in ast_df.groupby('clip_id')}
print('\nAST event rows:', len(ast_df))
matched_mask = ast_df['event_norm'].isin(RULE_LOOKUP.keys())
print('Rows matched by ontology:', int(matched_mask.sum()))



CLIP DEVELOPMENT-SET CHECK
Rows: 6000
Unique clips: 6000

Class distribution:
ground_truth
airport              600
bus                  600
metro                600
metro station        600
park                 600
public square        600
shopping mall        600
pedestrian street    600
street traffic       600
tram                 600
Name: count, dtype: int64

AST event column: label
AST score column: max_score

AST / CLIP MATCH CHECK
CLIP clips: 6000
AST matched clips: 6000
Missing AST clips: 0

AST event rows: 97116
Rows matched by ontology: 26340


## 5. Same ontology helpers as the basic ontology notebook

In [13]:
def safe_float(value, default=0.0):
    try:
        value = float(value)
        if np.isfinite(value):
            return value
    except Exception:
        pass
    return default

# Ontology normalization
def normalize_ontology_scores(ontology_scores, candidates, mode='sum'):
    if mode == 'sum':
        denominator = sum((ontology_scores[scene] for scene in candidates))
    elif mode == 'max':
        denominator = max((ontology_scores[scene] for scene in candidates))
    else:
        raise ValueError("Normalization mode must be 'sum' or 'max'.")
    if denominator > 0:
        return {scene: ontology_scores[scene] / denominator for scene in candidates}
    return {scene: 0.0 for scene in candidates}

# CLIP protection gap
def compute_clip_protection_gap(clip_scores, candidates, proposed_winner, mode='candidate'):
    original_top1 = candidates[0]
    original_score = clip_scores[original_top1]
    if mode == 'candidate':
        comparison_scene = proposed_winner
    elif mode == 'top2':
        comparison_scene = candidates[1] if len(candidates) >= 2 else original_top1
    else:
        raise ValueError("CLIP protection mode must be 'candidate' or 'top2'.")
    comparison_score = clip_scores[comparison_scene]
    return max(0.0, original_score - comparison_score)


## 6. Evaluate one frozen-structure grid configuration

`run_ontology_for_clip()` below is the frozen CLIP + AST + ontology reasoning function. The grid changes only:

1. `min_event_confidence`
2. `ontology_lambda`
3. `clip_protection_margin`

No SUPPORT-count gate, separate REFINE gate, or other grid-specific decision layer is added.


In [14]:
# CLIP + AST + ontology for one CLIP
def run_ontology_for_clip(
    clip_row,
    audio_group,
    min_event_confidence,
    ontology_lambda,
    clip_protection_margin,
    normalization_mode='sum',
    clip_protection_mode='candidate',
    return_details=False,
):
    candidates = list(dict.fromkeys([str(clip_row['clip_top1']), str(clip_row['clip_top2']), str(clip_row['clip_top3'])]))
    original_top1 = candidates[0]
    clip_scores = {scene: safe_float(clip_row[CLIP_SCORE_COLUMNS[scene]]) for scene in candidates}
    ontology_scores = {scene: 0.0 for scene in candidates}
    support_scores = {scene: 0.0 for scene in candidates}
    refine_scores = {scene: 0.0 for scene in candidates}
    fired_rules = []
    ignored_events = []
    if audio_group is not None:
        for _, event_row in audio_group.iterrows():
            event = event_row['event_norm']
            confidence = safe_float(event_row['event_score'])
            persistence = safe_float(event_row['persistence'])
            if not np.isfinite(confidence) or confidence < min_event_confidence:
                continue
            rule = RULE_LOOKUP.get(event)
            if rule is None:
                continue
            relation = str(rule['relation']).upper()
            if relation == 'IGNORE':
                if return_details:
                    ignored_events.append(f'{event}({confidence:.3f})')
                continue
            for target_scene, semantic_strength in rule['targets'].items():
                if target_scene not in candidates:
                    continue
                semantic_strength = safe_float(semantic_strength)
                evidence = confidence * persistence * semantic_strength
                ontology_scores[target_scene] += evidence
                if relation == 'SUPPORT':
                    support_scores[target_scene] += evidence
                elif relation == 'REFINE':
                    refine_scores[target_scene] += evidence
                if return_details:
                    fired_rules.append(f"{rule['name']}:{event}->{target_scene}[{relation};conf={confidence:.3f};pers={persistence:.2f};sem={semantic_strength:.2f};E={evidence:.3f}]")
    ontology_normalized = normalize_ontology_scores(
        ontology_scores=ontology_scores,
        candidates=candidates,
        mode=normalization_mode,
    )
    adjusted_scores = {scene: clip_scores[scene] + ontology_lambda * ontology_normalized[scene] for scene in candidates}
    proposed_winner = max(candidates, key=lambda scene: adjusted_scores[scene])
    clip_gap = compute_clip_protection_gap(
        clip_scores=clip_scores,
        candidates=candidates,
        proposed_winner=proposed_winner,
        mode=clip_protection_mode,
    )
    original_evidence = ontology_scores[original_top1]
    alternative_evidence = ontology_scores[proposed_winner]
    ontology_advantage = alternative_evidence - original_evidence
    final_prediction = original_top1
    decision = 'KEEP_CLIP'
    reason = 'CLIP Top-1 retained'
    if proposed_winner != original_top1:
        enough_ontology = alternative_evidence >= MIN_ONTOLOGY_EVIDENCE
        enough_advantage = ontology_advantage >= MIN_ONTOLOGY_ADVANTAGE
        visual_gap_allowed = clip_gap <= clip_protection_margin
        strong_audio = alternative_evidence >= STRONG_ONTOLOGY_EVIDENCE
        if enough_ontology and enough_advantage and (visual_gap_allowed or strong_audio):
            final_prediction = proposed_winner
            decision = 'ONTOLOGY_RERANK'
            if strong_audio:
                reason = 'Strong ontology evidence overrode CLIP protection margin'
            else:
                reason = 'Ontology evidence corrected uncertain CLIP Top-1'
        elif not enough_ontology:
            reason = 'Alternative ontology evidence below minimum'
        elif not enough_advantage:
            reason = 'Ontology advantage below minimum'
        elif not (visual_gap_allowed or strong_audio):
            reason = 'CLIP protection margin blocked correction'
    output = {
        'ontology_prediction': final_prediction,
        'ontology_decision': decision,
        'ontology_reason': reason,
        'candidate_after_fusion': proposed_winner,
        'clip_margin_to_candidate': clip_gap,
        'ontology_advantage': ontology_advantage,
        'current_ontology_evidence': original_evidence,
        'alternative_ontology_evidence': alternative_evidence,
        'min_event_confidence': float(min_event_confidence),
        'ontology_lambda': float(ontology_lambda),
        'clip_protection_margin': float(clip_protection_margin),
    }
    if return_details:
        output.update({'fired_rule_count': len(fired_rules), 'fired_rules': ' | '.join(fired_rules), 'ignored_events': ' | '.join(ignored_events)})
        for candidate_index in range(3):
            n = candidate_index + 1
            if candidate_index < len(candidates):
                scene = candidates[candidate_index]
                output[f'candidate{n}'] = scene
                output[f'candidate{n}_clip_score'] = clip_scores[scene]
                output[f'candidate{n}_ontology_raw'] = ontology_scores[scene]
                output[f'candidate{n}_ontology_norm'] = ontology_normalized[scene]
                output[f'candidate{n}_support'] = support_scores[scene]
                output[f'candidate{n}_refine'] = refine_scores[scene]
                output[f'candidate{n}_adjusted'] = adjusted_scores[scene]
            else:
                output[f'candidate{n}'] = np.nan
                output[f'candidate{n}_clip_score'] = np.nan
                output[f'candidate{n}_ontology_raw'] = np.nan
                output[f'candidate{n}_ontology_norm'] = np.nan
                output[f'candidate{n}_support'] = np.nan
                output[f'candidate{n}_refine'] = np.nan
                output[f'candidate{n}_adjusted'] = np.nan
    return output

# Evaluate one grid configuration
def run_configuration(
    min_event_confidence,
    ontology_lambda,
    clip_protection_margin,
    return_predictions=False,
):
    predictions = []
    detailed_rows = []
    for _, clip_row in clip_df.iterrows():
        clip_id = str(clip_row['clip_id'])
        audio_group = ast_groups.get(clip_id)
        result = run_ontology_for_clip(
            clip_row=clip_row,
            audio_group=audio_group,
            min_event_confidence=min_event_confidence,
            ontology_lambda=ontology_lambda,
            clip_protection_margin=clip_protection_margin,
            normalization_mode=NORMALIZATION_MODE,
            clip_protection_mode=CLIP_PROTECTION_MODE,
            return_details=return_predictions,
        )
        final_prediction = str(result['ontology_prediction'])
        predictions.append(final_prediction)
        if return_predictions:
            detailed_rows.append({'clip_id': clip_id, 'ground_truth': str(clip_row['ground_truth']), 'clip_top1': str(clip_row['clip_top1']), 'clip_top2': str(clip_row['clip_top2']), 'clip_top3': str(clip_row['clip_top3']), 'candidate_after_fusion': result['candidate_after_fusion'], 'final_prediction': final_prediction, 'decision': result['ontology_decision'], 'reason': result['ontology_reason'], 'clip_margin_to_candidate': result['clip_margin_to_candidate'], 'ontology_advantage': result['ontology_advantage'], 'alternative_ontology_evidence': result['alternative_ontology_evidence'], 'original_ontology_evidence': result['current_ontology_evidence'], 'min_event_confidence': float(min_event_confidence), 'ontology_lambda': float(ontology_lambda), 'clip_protection_margin': float(clip_protection_margin), 'fired_rules': result.get('fired_rules', '')})
    y_true = clip_df['ground_truth'].astype(str).tolist()
    clip_predictions = clip_df['clip_top1'].astype(str).tolist()
    accuracy = accuracy_score(y_true, predictions)
    macro_f1 = f1_score(y_true, predictions, labels=TAU_CLASSES, average='macro', zero_division=0)
    clip_correct = np.array([clip_pred == truth for clip_pred, truth in zip(clip_predictions, y_true)])
    final_correct = np.array([final_pred == truth for final_pred, truth in zip(predictions, y_true)])
    changed = np.array([final_pred != clip_pred for final_pred, clip_pred in zip(predictions, clip_predictions)])
    wrong_to_correct = int(np.sum(changed & ~clip_correct & final_correct))
    correct_to_wrong = int(np.sum(changed & clip_correct & ~final_correct))
    wrong_to_wrong = int(np.sum(changed & ~clip_correct & ~final_correct))
    net_corrections = wrong_to_correct - correct_to_wrong
    correction_attempts = wrong_to_correct + correct_to_wrong
    correction_precision = wrong_to_correct / correction_attempts if correction_attempts > 0 else 0.0
    changed_count = int(changed.sum())
    beneficial_change_rate = wrong_to_correct / changed_count if changed_count > 0 else 0.0
    metric_result = {
        'min_event_confidence': float(min_event_confidence),
        'ontology_lambda': float(ontology_lambda),
        'clip_protection_margin': float(clip_protection_margin),
        'accuracy_percent': float(accuracy * 100),
        'macro_f1': float(macro_f1),
        'wrong_to_correct': wrong_to_correct,
        'correct_to_wrong': correct_to_wrong,
        'wrong_to_wrong': wrong_to_wrong,
        'net_corrections': net_corrections,
        'correction_precision_percent': float(correction_precision * 100),
        'beneficial_change_rate_percent': float(beneficial_change_rate * 100),
        'predictions_changed': changed_count,
    }
    if return_predictions:
        return (metric_result, pd.DataFrame(detailed_rows))
    return metric_result


## 7. Run all 18 frozen-structure configurations


In [15]:
print('\nTotal configurations:', len(GRID_SETTINGS))
print('=' * 110)
grid_results = []
for config_number, (
    min_event_confidence,
    ontology_lambda,
    clip_protection_margin) in enumerate(GRID_SETTINGS,
    start=1,
):
    result = run_configuration(
        min_event_confidence=min_event_confidence,
        ontology_lambda=ontology_lambda,
        clip_protection_margin=clip_protection_margin,
        return_predictions=False,
    )
    grid_results.append(result)
    print(f"[{config_number:02d}/{len(GRID_SETTINGS):02d}] EVENT={min_event_confidence:.2f} | LAMBDA={ontology_lambda:.2f} | MARGIN={clip_protection_margin:.2f} => Acc={result['accuracy_percent']:.4f}% | F1={result['macro_f1']:.6f} | W→C={result['wrong_to_correct']} | C→W={result['correct_to_wrong']} | W→W={result['wrong_to_wrong']} | Net={result['net_corrections']:+d}")
leaderboard = pd.DataFrame(grid_results)

# Accuracy rank
accuracy_ranked = leaderboard.sort_values(
    by=['accuracy_percent', 'macro_f1', 'correct_to_wrong', 'net_corrections'],
    ascending=[False, False, True, False]).reset_index(drop=True,
)
accuracy_ranked['accuracy_rank'] = np.arange(len(accuracy_ranked)) + 1

# Safe-net rank
safe_ranked = leaderboard.sort_values(
    by=['net_corrections', 'correct_to_wrong', 'accuracy_percent', 'macro_f1'],
    ascending=[False, True, False, False]).reset_index(drop=True,
)
safe_ranked['safe_rank'] = np.arange(len(safe_ranked)) + 1

# Combine ranks
leaderboard_final = accuracy_ranked.merge(
    safe_ranked[['min_event_confidence', 'ontology_lambda', 'clip_protection_margin', 'safe_rank']],
    on=['min_event_confidence', 'ontology_lambda', 'clip_protection_margin'],
    how='left',
)
leaderboard_final.to_csv(LEADERBOARD_PATH, index=False)
print('\n' + '=' * 125)
print('FULL FROZEN-STRUCTURE GRID SEARCH LEADERBOARD — ACCURACY RANK')
print('=' * 125)
display_columns = [
    'accuracy_rank',
    'safe_rank',
    'min_event_confidence',
    'ontology_lambda',
    'clip_protection_margin',
    'accuracy_percent',
    'macro_f1',
    'wrong_to_correct',
    'correct_to_wrong',
    'wrong_to_wrong',
    'net_corrections',
    'correction_precision_percent',
    'beneficial_change_rate_percent',
    'predictions_changed',
]
print(leaderboard_final[display_columns].round(6).to_string(index=False))
best_accuracy = accuracy_ranked.iloc[0]
best_safe = safe_ranked.iloc[0]
print('\n' + '=' * 100)
print('BEST CONFIGURATION — ACCURACY')
print('=' * 100)
print(best_accuracy.to_string())
print('\n' + '=' * 100)
print('BEST CONFIGURATION — SAFE NET')
print('=' * 100)
print(best_safe.to_string())



Total configurations: 120
[01/120] EVENT=0.05 | LAMBDA=0.15 | MARGIN=0.10 => Acc=79.3500% | F1=0.791979 | W→C=28 | C→W=3 | W→W=2 | Net=+25
[02/120] EVENT=0.05 | LAMBDA=0.15 | MARGIN=0.15 => Acc=79.3667% | F1=0.792133 | W→C=29 | C→W=3 | W→W=3 | Net=+26
[03/120] EVENT=0.05 | LAMBDA=0.15 | MARGIN=0.20 => Acc=79.3667% | F1=0.792133 | W→C=29 | C→W=3 | W→W=3 | Net=+26
[04/120] EVENT=0.05 | LAMBDA=0.15 | MARGIN=0.25 => Acc=79.3667% | F1=0.792133 | W→C=29 | C→W=3 | W→W=3 | Net=+26
[05/120] EVENT=0.05 | LAMBDA=0.15 | MARGIN=0.30 => Acc=79.3667% | F1=0.792133 | W→C=29 | C→W=3 | W→W=3 | Net=+26
[06/120] EVENT=0.05 | LAMBDA=0.20 | MARGIN=0.10 => Acc=79.4167% | F1=0.792698 | W→C=33 | C→W=4 | W→W=2 | Net=+29
[07/120] EVENT=0.05 | LAMBDA=0.20 | MARGIN=0.15 => Acc=79.4500% | F1=0.793012 | W→C=35 | C→W=4 | W→W=3 | Net=+31
[08/120] EVENT=0.05 | LAMBDA=0.20 | MARGIN=0.20 => Acc=79.4500% | F1=0.793012 | W→C=35 | C→W=4 | W→W=3 | Net=+31
[09/120] EVENT=0.05 | LAMBDA=0.20 | MARGIN=0.25 => Acc=79.4500% | F1=

## 8. Save predictions for the selected frozen-structure development configuration


In [16]:
# Choose which ranking defines the development-selected configuration.
# "accuracy" = highest development accuracy
# "safe_net" = highest net correction with fewer harmful changes
# Keep "accuracy" unless your experimental protocol explicitly
# specifies another selection rule.
FINAL_SELECTION = 'accuracy'
if FINAL_SELECTION == 'safe_net':
    selected = best_safe
else:
    selected = best_accuracy
selected_event_confidence = float(selected['min_event_confidence'])
selected_lambda = float(selected['ontology_lambda'])
selected_margin = float(selected['clip_protection_margin'])
best_metrics, best_predictions = run_configuration(
    min_event_confidence=selected_event_confidence,
    ontology_lambda=selected_lambda,
    clip_protection_margin=selected_margin,
    return_predictions=True,
)

# Change types
best_predictions['clip_correct'] = best_predictions['clip_top1'] == best_predictions['ground_truth']
best_predictions['final_correct'] = best_predictions['final_prediction'] == best_predictions['ground_truth']
best_predictions['changed'] = best_predictions['clip_top1'] != best_predictions['final_prediction']
best_predictions[
    'change_type'] = np.select([best_predictions['changed'] & ~best_predictions['clip_correct'] & best_predictions['final_correct'], best_predictions['changed'] & best_predictions['clip_correct'] & ~best_predictions['final_correct'], best_predictions['changed'] & ~best_predictions['clip_correct'] & ~best_predictions['final_correct']],
    ['Wrong->Correct', 'Correct->Wrong', 'Wrong->Wrong'],
    default='UNCHANGED',
)
best_predictions.to_csv(BEST_PREDICTIONS_PATH, index=False)

# Per-class results
per_class_rows = []
for scene in TAU_CLASSES:
    temp = best_predictions[best_predictions['ground_truth'] == scene].copy()
    clip_acc = (temp['clip_top1'] == scene).mean()
    final_acc = (temp['final_prediction'] == scene).mean()
    per_class_rows.append({'class': scene, 'number_of_clips': len(temp), 'clip_accuracy_percent': float(clip_acc * 100), 'selected_accuracy_percent': float(final_acc * 100), 'change_points': float((final_acc - clip_acc) * 100), 'wrong_to_correct': int((temp['change_type'] == 'Wrong->Correct').sum()), 'correct_to_wrong': int((temp['change_type'] == 'Correct->Wrong').sum()), 'wrong_to_wrong': int((temp['change_type'] == 'Wrong->Wrong').sum())})
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(BEST_PER_CLASS_PATH, index=False)

# Save config
best_config = {
    'experiment': 'development_6000_frozen_ontology_grid_search',
    'selection_method': FINAL_SELECTION,
    'min_event_confidence': selected_event_confidence,
    'ontology_lambda': selected_lambda,
    'clip_protection_margin': selected_margin,
    'min_ontology_evidence': MIN_ONTOLOGY_EVIDENCE,
    'min_ontology_advantage': MIN_ONTOLOGY_ADVANTAGE,
    'strong_ontology_evidence': STRONG_ONTOLOGY_EVIDENCE,
    'ontology_normalization': NORMALIZATION_MODE,
    'clip_protection_mode': CLIP_PROTECTION_MODE,
    'ontology_source': '../Acoustic_ontology.ipynb',
    **best_metrics,
}
with open(BEST_CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(best_config, f, indent=2)
print('\n' + '=' * 100)
print('SELECTED DEVELOPMENT CONFIGURATION')
print('=' * 100)
print('Selection method:', FINAL_SELECTION)
print(f'Event confidence:       {selected_event_confidence:.2f}')
print(f'Ontology lambda:        {selected_lambda:.2f}')
print(f'CLIP protection margin: {selected_margin:.2f}')
print(f"Accuracy:               {best_metrics['accuracy_percent']:.4f}%")
print(f"Macro-F1:               {best_metrics['macro_f1']:.6f}")
print(f"W→C / C→W / W→W:       {best_metrics['wrong_to_correct']} / {best_metrics['correct_to_wrong']} / {best_metrics['wrong_to_wrong']}")
print(f"Net corrections:        {best_metrics['net_corrections']:+d}")
print(f"Correction precision:   {best_metrics['correction_precision_percent']:.2f}%")
print('\nSaved:')
print(LEADERBOARD_PATH)
print(BEST_PREDICTIONS_PATH)
print(BEST_PER_CLASS_PATH)
print(BEST_CONFIG_PATH)



SELECTED DEVELOPMENT CONFIGURATION
Selection method: accuracy
Event confidence:       0.08
Ontology lambda:        0.40
CLIP protection margin: 0.20
Accuracy:               79.7333%
Macro-F1:               0.795959
W→C / C→W / W→W:       51 / 3 / 4
Net corrections:        +48
Correction precision:   94.44%

Saved:
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_GRID_SEARCH_6000\ontology_grid_search_18_settings.csv
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_GRID_SEARCH_6000\ontology_grid_search_best_predictions.csv
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_GRID_SEARCH_6000\ontology_grid_search_best_per_class.csv
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_GRID_SEARCH_6000\ontology_grid_search_best_config.json
